# Prepare Electron Microscopy Data

The powerfit program requires a EM density of a unknown structure where it can fit structures into.

Most of the the EM density contains multiple structures, so we want to remove all but the unknown structure.

In this example we will use the [phenix software](https://www.phenix-online.org/) to prepare the EM density.
Make sure you have phenix installed and its commands like `phenix.about` are callable from the command line.

For this example we will use [EMD-33292](https://www.ebi.ac.uk/emdb/EMD-33292), a sodium channel, with fitted model [7xm9](https://www.ebi.ac.uk/pdbe/entry/pdb/7xm9).

The fitted model consist of following chains:
- A: Isoform 3 of Sodium channel protein type 9 subunit alpha,Green fluorescent protein
- B: Sodium channel subunit beta-1,Green fluorescent protein
- C: Sodium channel subunit beta-2

We will use the B chain, the sodium channel subunit beta-1, as the unknown structure.

Lets start by downloading the density map and the fitted model.

In [1]:
!wget -nc https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-33292/map/emd_33292.map.gz
!gunzip emd_33292.map.gz
!wget -nc https://www.ebi.ac.uk/pdbe/entry-files/download/7xm9.cif

--2025-07-04 08:22:06--  https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-33292/map/emd_33292.map.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 62515467 (60M) [application/x-gzip]
Saving to: ‘emd_33292.map.gz’

emd_33292.map.gz    100%[===================>]  59.62M   680KB/s    in 98s     

2025-07-04 08:23:46 (625 KB/s) - ‘emd_33292.map.gz’ saved [62515467/62515467]

--2025-07-04 08:23:47--  https://www.ebi.ac.uk/pdbe/entry-files/download/7xm9.cif
Resolving www.ebi.ac.uk (www.ebi.ac.uk)... 193.62.193.80
Connecting to www.ebi.ac.uk (www.ebi.ac.uk)|193.62.193.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1569626 (1.5M) [text/plain]
Saving to: ‘7xm9.cif’

7xm9.cif            100%[===================>]   1.50M  9.39MB/s    in 0.2s    

2025-07-04 08:23:48 (9.39 MB/s) - ‘7xm9.cif’ saved [1569626/1569626]



In [ ]:
# TODO visualize with molviewspec in Mol*

Phenix has a command called [phenix.map_box](https://phenix-online.org/documentation/reference/map_box.html) that can be used to mask out the known model (chain A and C) from the EM density and keep the unknown structure (chain B).

In [4]:
from pathlib import Path

Path("map_box.def").write_text("""
selection = chain B
mask_atoms = true
resolution = 3
soft_mask = true
increase_box_cushion_and_atom_radius_for_soft_mask = false
output_file_name_prefix = 7xm9-B
""")

163

In [8]:
!/home/verhoes/git/protein-detective/phenix/phenix-1.21.2-5419/phenix_bin/phenix.map_box 7xm9.cif emd_33292.map map_box.def

job_title = None
pdb_file = None
map_coefficients_file = None
label = None
ccp4_map_file = None
mask_file_name = None
target_ncs_au_file = None
selection = "chain B"
selection_radius = 3
box_cushion = 3
mask_atoms = True
mask_atoms_atom_radius = 3
write_mask_file = False
set_outside_to_mean_inside = False
resolution_factor = 0.25
map_scale_factor = None
scale_max = 99999
resolution = 3
output_format = xplor *mtz *ccp4
output_file_name_prefix = "7xm9-B"
mask_select = False
density_select = False
density_select_threshold = 0.05
get_half_height_width = True
symmetry = None
symmetry_file = None
sequence_file = None
molecular_mass = None
solvent_content = None
extract_unique = False
increase_box_cushion_and_atom_radius_for_soft_mask = False
soft_mask_extract_unique = True
mask_expand_ratio = 1
regions_to_keep = None
keep_low_density = True
chain_type = None *PROTEIN DNA RNA
soft_mask = True
invert_mask = False
soft_mask_radius = 3
lower_bounds = None
upper_bounds = None
bounds_are_absolute 

To run powerfit we need to know the resolution of the EM density, which can be retrieved using [phenix.mtriage](https://phenix-online.org/documentation/reference/mtriage.html)

In [9]:
!/home/verhoes/git/protein-detective/phenix/phenix-1.21.2-5419/phenix_bin/phenix.mtriage 7xm9-B.ccp4 write_mask_file=False |grep Resolution


Resolution set to 3.48 A 


We can now run powerfit on 7xm9-B.ccp4 as target density map and 7xm9.cif as template.


In [10]:
!powerfit 7xm9-B.ccp4 3.48 7xm9.cif -d powerfit-7xm9-B -c B

Target file read from:                                                          
/home/verhoes/git/protein-detective/protein-detective/docs/7xm9-B.ccp4          
Traceback (most recent call last):
  File "/home/verhoes/git/protein-detective/protein-detective/.venv/bin/powerfit", line 10, in <module>
    sys.exit(main())
             ~~~~^^
  File "/home/verhoes/git/protein-detective/protein-detective/.venv/lib/python3.13/site-packages/powerfit_em/powerfit.py", line 252, in main
    powerfit(
    ~~~~~~~~^
        target_volume=args.target,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<15 lines>...
        progress=progress
        ^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/verhoes/git/protein-detective/protein-detective/.venv/lib/python3.13/site-packages/powerfit_em/powerfit.py", line 314, in powerfit
    target = Volume.fromfile(target_volume)
  File "/home/verhoes/git/protein-detective/protein-detective/.venv/lib/python3.13/site-packages/powerfit_em/volume.py", line 17, in fromfile
   